In [6]:
# === Einstellungen ===
DATA_DIR = "data"               # Hauptordner mit train/val/test
TRAIN_DIR = f"{DATA_DIR}/train"
VAL_DIR   = f"{DATA_DIR}/val"
TEST_DIR  = f"{DATA_DIR}/test"

# EfficientNetB3 erwartet normalerweise 300x300, passe an falls nötig
IMG_SIZE = (224, 224)          # falls dein von-scratch CNN andere Größe hat: hier anpassen
BATCH_SIZE = 32
SEED = 42
EPOCHS_HEAD = 30                # Kopf-Training
EPOCHS_FINETUNE = 8            # Fine-Tuning (optional)
AUTOTUNE = None

SCRATCH_MODEL_PATH = "scratch_model.h5"   # Pfad, falls du bereits ein Scratch-Modell hast
TL_MODEL_SAVE = "transfer_model.h5"
METRICS_SAVE = "metrics_transfer.json"


In [7]:
# === Imports ===
import os, json, datetime
import numpy as np
import matplotlib.pyplot as plt
import itertools
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow version:", tf.__version__)


2025-12-07 23:10:06.269242: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.12.0


In [9]:
# 1) Split & Kopieren aus deinem PlantVillage-Ordner
import shutil, random
from pathlib import Path
from collections import defaultdict

random.seed(42)

SOURCE_ROOT = Path("PlantVillage")   # <- das ist dein existierender Ordner
TARGET_ROOT = Path("data")           # <- Zielstruktur wird hier angelegt
TRAIN_DIR = TARGET_ROOT / "train"
VAL_DIR   = TARGET_ROOT / "val"
TEST_DIR  = TARGET_ROOT / "test"

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"{SOURCE_ROOT} existiert nicht. Prüfe den Pfad.")

# Bildendungen akzeptieren
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

# Klassen sind die direkten Unterordner von SOURCE_ROOT
classes = [p.name for p in SOURCE_ROOT.iterdir() if p.is_dir()]
if not classes:
    raise ValueError(f"Keine Klassenordner unter {SOURCE_ROOT} gefunden.")

print(f"Gefundene Klassen: {classes[:20]} (insgesamt {len(classes)})")

# Erstelle Zielordner
for base in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    base.mkdir(parents=True, exist_ok=True)

# Split-Raten
ratio_train = 0.70
ratio_val = 0.15
ratio_test = 0.15

total_before = 0
total_after = 0

for cls in classes:
    src_cls_dir = SOURCE_ROOT / cls
    imgs = [p for p in src_cls_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    if len(imgs) == 0:
        print(f"Warnung: Klasse {cls} enthält keine Bilder, wird übersprungen.")
        continue
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * ratio_train)
    n_val = int(n * ratio_val)
    n_test = n - n_train - n_val

    splits = {
        TRAIN_DIR / cls: imgs[:n_train],
        VAL_DIR / cls:   imgs[n_train:n_train+n_val],
        TEST_DIR / cls:  imgs[n_train+n_val:]
    }

    total_before += n
    for target_dir, files in splits.items():
        target_dir.mkdir(parents=True, exist_ok=True)
        for src in files:
            dst = target_dir / src.name
            # falls identische Datei schon existiert -> umbenennen
            if dst.exists():
                dst = target_dir / f"{dst.stem}_{random.randint(0,99999)}{dst.suffix}"
            shutil.copy2(src, dst)
            total_after += 1

print(f"Fertig. Insgesamt Bilder (Quelle): {total_before}, kopierte Bilder (Ziel): {total_after}")
for d in ("train", "val", "test"):
    cnt = sum(1 for _ in (TARGET_ROOT / d).rglob("*") if _.suffix.lower() in IMG_EXTS)
    print(f"  {d}: {cnt} Bilder")


Gefundene Klassen: ['Tomato_healthy', 'Potato___Early_blight', 'PlantVillage', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Early_blight', 'Tomato__Target_Spot', 'Potato___Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato_Septoria_leaf_spot', 'Tomato__Tomato_mosaic_virus', 'Pepper__bell___Bacterial_spot', 'Tomato_Bacterial_spot', 'Tomato_Late_blight', 'Pepper__bell___healthy', 'Potato___healthy'] (insgesamt 16)


In [ ]:
# 2) Validierung: zeige Klassen und ein paar Dateien pro Klasse
from pathlib import Path

TARGET_ROOT = Path("data")
TRAIN_DIR = TARGET_ROOT / "train"
classes_target = [p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]
print("Ziel-Klassen (train):", classes_target[:30], f"(insgesamt {len(classes_target)})")

# Zeige für 5 Klassen je bis zu 5 Dateien
print("\nBeispiele:")
count = 0
for cls in classes_target[:5]:
    files = list((TRAIN_DIR/cls).glob("*"))
    print(f"- {cls}: {len(files)} files, Beispiele:", [f.name for f in files[:5]])
    count += 1


In [ ]:
# 3) Dataset laden (nun sollte data/train existieren)
import tensorflow as tf

DATA_DIR = "data"
TRAIN_DIR = f"{DATA_DIR}/train"
VAL_DIR   = f"{DATA_DIR}/val"
TEST_DIR  = f"{DATA_DIR}/test"

IMG_SIZE = (300, 300)   # anpassen falls nötig (sollte mit deinem TL-Setup übereinstimmen)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
    seed=SEED
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:  ", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches: ", tf.data.experimental.cardinality(test_ds).numpy())

# Prefetch für Performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# === Daten laden (multiclass) ===
# Erwartete Struktur:
# data/train/class_x/*.jpg
# data/val/class_x/*.jpg
# data/test/class_x/*.jpg

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",   # multiclass -> categorical
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
    seed=SEED
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

# Performance
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


NotFoundError: Could not find directory data/train

In [ ]:
# === Data Augmentation (wie gewünscht: Rotation, Kontrast, Flip, Zoom, Translation) ===
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),         # Rotation
    layers.RandomZoom(0.12),             # Zoom
    layers.RandomTranslation(0.08, 0.08),# Translation
    layers.RandomContrast(0.15),         # Kontrast
], name="data_augmentation")


In [ ]:
# === Model Builder: EfficientNetB3 backbone ===
def build_transfer_model(base_model_name="EfficientNetB3",
                         input_shape=IMG_SIZE + (3,),
                         num_classes=NUM_CLASSES,
                         train_base=False,
                         dropout_rate=0.3):
    if base_model_name == "EfficientNetB3":
        base = keras.applications.EfficientNetB3(include_top=False,
                                                 input_shape=input_shape,
                                                 weights="imagenet")
        preprocess = keras.applications.efficientnet.preprocess_input
    else:
        raise ValueError("Nur EfficientNetB3 ist in dieser Zelle vorgesehen. Passe bei Bedarf an.")

    base.trainable = train_base  # initial: nur Kopf trainierbar

    inputs = keras.Input(shape=input_shape)
    x = inputs
    x = data_augmentation(x)                     # augmentation on the fly
    x = layers.Lambda(preprocess, name="preprocess")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=f"TL_{base_model_name}")
    return model

# Test: Model initialisieren und summary anzeigen
tl_model = build_transfer_model(train_base=False)
tl_model.summary()


In [ ]:
# === Kompilieren & Kopf trainieren (Phase 1) ===
loss = "categorical_crossentropy" if NUM_CLASSES > 1 else "binary_crossentropy"

tl_model = build_transfer_model(train_base=False)
tl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                 loss=loss,
                 metrics=["accuracy"])

callbacks = [
    keras.callbacks.ModelCheckpoint("tl_best_head.h5", save_best_only=True, monitor="val_accuracy"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history_head = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


In [ ]:
# === Fine-Tuning: einige Basis-Layer trainierbar machen ===
# Hier: letzte 30% der Basis-Layer freigeben (als Beispiel)
base = None
# suche das Base-Model-Objekt in tl_model (robuster Zugriff)
for layer in tl_model.layers:
    if "efficientnet" in layer.name:
        base = layer
        break
if base is None:
    # fallback: oft liegt das base model an index 3
    base = tl_model.layers[3]

print("Base model:", base.name)
base.trainable = True

# freeze first x% layers of base
n_layers = len(base.layers)
freeze_until = int(0.7 * n_layers)  # friere die ersten 70%, taile die anderen
for i, layer in enumerate(base.layers):
    layer.trainable = False if i < freeze_until else True

tl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
                 loss=loss,
                 metrics=["accuracy"])

callbacks_ft = [
    keras.callbacks.ModelCheckpoint(TL_MODEL_SAVE, save_best_only=True, monitor="val_accuracy"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history_ft = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks_ft
)


In [ ]:
# === Evaluation auf Testdaten & Plots ===
# Lade bestes gespeichertes Modell falls vorhanden
if os.path.exists(TL_MODEL_SAVE):
    tl_model = keras.models.load_model(TL_MODEL_SAVE)
    print("Loaded saved TL model:", TL_MODEL_SAVE)

tl_eval = tl_model.evaluate(test_ds)
print("TL model evaluation (loss, acc):", tl_eval)

# Vorhersagen sammeln
y_true = []
y_pred = []
for images, labels in test_ds:
    preds = tl_model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# sklearn Reports
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
cm = confusion_matrix(y_true, y_pred)
print("Test accuracy (sklearn):", accuracy_score(y_true, y_pred))
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot History helper
def plot_history(h_head, h_ft=None):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(h_head.history.get("accuracy", []), label="train_head")
    plt.plot(h_head.history.get("val_accuracy", []), label="val_head")
    if h_ft:
        plt.plot(h_ft.history.get("accuracy", []), label="train_ft")
        plt.plot(h_ft.history.get("val_accuracy", []), label="val_ft")
    plt.title("Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(h_head.history.get("loss", []), label="train_head")
    plt.plot(h_head.history.get("val_loss", []), label="val_head")
    if h_ft:
        plt.plot(h_ft.history.get("loss", []), label="train_ft")
        plt.plot(h_ft.history.get("val_loss", []), label="val_ft")
    plt.title("Loss")
    plt.legend()
    plt.show()

plot_history(history_head, history_ft if 'history_ft' in globals() else None)

# Confusion Matrix plot
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix'):
    plt.figure(figsize=(8,8))
    cm_plot = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] if normalize else cm
    plt.imshow(cm_plot, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha="right")
    plt.yticks(tick_marks, classes)
    thresh = cm_plot.max() / 2.
    for i, j in itertools.product(range(cm_plot.shape[0]), range(cm_plot.shape[1])):
        plt.text(j, i, f"{cm[i, j]}" if not normalize else f"{cm_plot[i, j]:.2f}",
                 horizontalalignment="center",
                 color="white" if cm_plot[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

plot_confusion_matrix(cm, class_names, normalize=False, title="Confusion matrix (counts)")
